In [ ]:
from datetime import datetime

from pyspark.sql import SparkSession  # type: ignore

### 1. SparkSessionの初期化と接続設定 

In [8]:
# SparkSessionの作成（Cassandraコネクタのパッケージと接続先を指定）
spark = (
    SparkSession.builder.appName("JupyterCassandraIntegration")
    .config("spark.cassandra.connection.host", "cassandra")
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
    .getOrCreate()
)

print("Spark Version:", spark.version)
print("Spark Config Packages:", spark.conf.get("spark.jars.packages", "Not Set"))

Spark Version: 3.5.0
Spark Config Packages: com.datastax.spark:spark-cassandra-connector_2.12:3.5.1


### 2. Cassandraからデータを読み込む (Read)

In [9]:
# ※お使いのCassandraのキースペース名（Keyspace）とテーブル名に書き換えてください
keyspace_name = "iot_keyspace"
table_name = "sensor_metrics_summary"

# CassandraのテーブルからデータをDataFrameとして読み込む
df = spark.read.format("org.apache.spark.sql.cassandra").options(table=table_name, keyspace=keyspace_name).load()

# スキーマ（列構造）の確認
df.printSchema()

# データの先頭5行を表示
df.show(5, truncate=False)

root
 |-- sensor_id: string (nullable = false)
 |-- period_type: string (nullable = false)
 |-- window_start: timestamp (nullable = true)
 |-- metric_name: string (nullable = true)
 |-- metric_value: double (nullable = true)

+---------+-----------+-------------------+------------------+------------------+
|sensor_id|period_type|window_start       |metric_name       |metric_value      |
+---------+-----------+-------------------+------------------+------------------+
|sensor_1 |daily      |2026-11-10 00:00:00|avg_temperature   |24.857039950021374|
|sensor_1 |daily      |2026-11-10 00:00:00|median_temperature|24.769614387879905|
|sensor_1 |daily      |2026-11-10 00:00:00|p95_temperature   |29.571582436919343|
|sensor_1 |daily      |2026-11-10 00:00:00|stddev_temperature|2.9441226064355286|
|sensor_1 |daily      |2026-11-10 00:00:00|total_count       |209.0             |
+---------+-----------+-------------------+------------------+------------------+
only showing top 5 rows



### 3. Cassandraへデータを書き込む (Write)

In [11]:
# サンプル用のDataFrameを作成する場合

specified_date = datetime(2026, 11, 10, 21, 11, 0)

data = [
    ("sensor_xx", "dayly", specified_date, "test_value_1", 1.001),
    ("sensor_xx", "dayly", specified_date, "test_value_2", 2.02),
    ("sensor_xx", "dayly", specified_date, "test_value_3", 3.0),
]
columns = ["sensor_id", "period_type", "window_start", "metric_name", "metric_value"]
new_df = spark.createDataFrame(data, columns)

# Cassandraのテーブルへ書き込み（追加書き込みの場合は append、上書きの場合は overwrite）
new_df.write.format("org.apache.spark.sql.cassandra").options(
    table="sensor_metrics_summary", keyspace="iot_keyspace"
).mode("append").save()

print("Data written to Cassandra successfully!")

Data written to Cassandra successfully!


In [ ]:
# 登録したデータを再確認する。

# CassandraのテーブルからデータをDataFrameとして読み込む
df = (
    spark.read.format("org.apache.spark.sql.cassandra")
    .options(table="sensor_metrics_summary", keyspace="iot_keyspace")
    .load()
)

df.createOrReplaceTempView("v_sensor_metrics_summary")

x_sensor_id = "sensor_xx"

df_metrics = spark.sql(
    """
    SELECT
        sensor_id,
        period_type,
        window_start,
        metric_name,
        metric_value
    FROM
        v_sensor_metrics_summary
    WHERE
        sensor_id = :v_sensor_id
    """,
    {"v_sensor_id": x_sensor_id},
)

df_metrics.show(20, truncate=False)

+---------+-----------+-------------------+------------+------------+
|sensor_id|period_type|window_start       |metric_name |metric_value|
+---------+-----------+-------------------+------------+------------+
|sensor_xx|dayly      |2026-11-10 21:11:00|test_value_1|1.001       |
|sensor_xx|dayly      |2026-11-10 21:11:00|test_value_2|2.02        |
|sensor_xx|dayly      |2026-11-10 21:11:00|test_value_3|3.0         |
+---------+-----------+-------------------+------------+------------+

